# StyleShift — Exploratory Data Analysis

This notebook covers:
1. **Class distribution** — how imbalanced is the dataset?
2. **Cosine similarity at boundaries** — does it drop at style changes?
3. **POS distribution differences** — same-author vs. different-author pairs
4. **Ablation results visualization**
5. **Document-level error analysis**

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '../features')
sys.path.insert(0, '../models')

RESULTS_DIR = Path('../results')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load precomputed features and labels
X_train_stylo = np.load(RESULTS_DIR / 'X_train_stylo.npy')
X_val_stylo   = np.load(RESULTS_DIR / 'X_val_stylo.npy')
X_train_emb   = np.load(RESULTS_DIR / 'X_train_emb.npy')
X_val_emb     = np.load(RESULTS_DIR / 'X_val_emb.npy')
y_train       = np.load(RESULTS_DIR / 'y_train.npy')
y_val         = np.load(RESULTS_DIR / 'y_val.npy')

with open(RESULTS_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

print(f'Train: {len(y_train)} pairs | Val: {len(y_val)} pairs')
print(f'Train changes: {y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Val changes:   {y_val.sum()} ({100*y_val.mean():.1f}%)')

## 1. Class Distribution (Imbalance Check)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, y, title in zip(axes, [y_train, y_val], ['Training Set', 'Validation Set']):
    counts = [int((y == 0).sum()), int((y == 1).sum())]
    bars = ax.bar(['Same author (Y=0)', 'Style change (Y=1)'], counts,
                  color=['#2196F3', '#E53935'], alpha=0.85)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}\n({100*count/len(y):.1f}%)', ha='center', va='bottom', fontsize=10)
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Class Imbalance — WHY accuracy is not a valid metric here', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_class_distribution.png', dpi=150)
plt.show()

print(f'\nImbalance ratio: {(y_train==0).sum()/(y_train==1).sum():.1f}:1 (no-change : change)')
print('A trivial "always predict no-change" classifier would achieve:',
      f'{100*(y_val==0).mean():.1f}% accuracy — but F1=0 on the minority class.')

## 2. Cosine Similarity at Boundaries

**Hypothesis:** Cosine similarity between consecutive blocks should be LOWER at real style change boundaries than within the same-author segments.

In [ ]:
# X_train_emb[:, 0] = cosine similarity for each pair
cosine_same   = X_train_emb[y_train == 0, 0]
cosine_change = X_train_emb[y_train == 1, 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cosine_same,   bins=40, alpha=0.6, color='#2196F3', label='Same author (Y=0)')
ax.hist(cosine_change, bins=40, alpha=0.6, color='#E53935', label='Style change (Y=1)')
ax.set_xlabel('Cosine Similarity between consecutive blocks')
ax.set_ylabel('Count')
ax.set_title('Cosine Similarity Distribution at Same-author vs. Change Boundaries')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_cosine_distribution.png', dpi=150)
plt.show()

print(f'Mean cosine (same author):  {cosine_same.mean():.4f} ± {cosine_same.std():.4f}')
print(f'Mean cosine (style change): {cosine_change.mean():.4f} ± {cosine_change.std():.4f}')
print(f'\nConclusion: {"✓ Lower cosine at change boundaries — hypothesis supported" if cosine_change.mean() < cosine_same.mean() else "✗ Hypothesis not supported"}')

## 3. POS Distribution Differences — Same vs. Different Author

In [ ]:
# Stylometric feature layout: [avg_word_length(1), ttr(1), punct(1), upper(1),
#                              POS_NOUN..DET(6), avg_sent_len(1), fk_grade(1), fog(1), fw(150)]
# POS features start at index 4
POS_TAGS = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'DET']
POS_START = 4

same_pos   = X_train_stylo[y_train == 0, POS_START:POS_START+6]  # abs differences at POS indices
change_pos = X_train_stylo[y_train == 1, POS_START:POS_START+6]

mean_same   = same_pos.mean(axis=0)
mean_change = change_pos.mean(axis=0)

x = np.arange(len(POS_TAGS))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, mean_same,   width, label='Same author (Y=0)', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, mean_change, width, label='Style change (Y=1)', color='#E53935', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(POS_TAGS)
ax.set_ylabel('Mean absolute difference in POS ratio')
ax.set_title('POS Tag Differences: Same Author vs. Style Change Pairs\n(higher = more dissimilar at that POS)')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_pos_differences.png', dpi=150)
plt.show()

## 4. Ablation Results

In [ ]:
ablation_path = RESULTS_DIR / 'ablation_results.json'
if not ablation_path.exists():
    print('Run: python evaluation/ablation.py')
else:
    with open(ablation_path) as f:
        ablation = json.load(f)

    variants = list(ablation.keys())
    # ablation_results.json now has a nested structure per variant:
    #   {variant: {default: {...}, tuned: {...}, per_difficulty: {easy/medium/hard: {...}}}}
    # We read the TUNED-threshold view — the operationally meaningful numbers.
    macro_f1s  = [ablation[v]['tuned']['macro_f1']  for v in variants]
    change_f1s = [ablation[v]['tuned']['change_f1'] for v in variants]
    auc_rocs   = [ablation[v]['tuned']['auc_roc']   for v in variants]

    abl_df = pd.DataFrame({'Variant': variants, 'Macro F1': macro_f1s,
                            'Change F1': change_f1s, 'AUC-ROC': auc_rocs})
    print(abl_df.to_string(index=False))

    x = np.arange(len(variants))
    width = 0.25
    short = ['Stream A\n(SVM)', 'Stream B\n(cosine)', 'Siamese\nalone', 'Full\nEnsemble']

    fig, ax = plt.subplots(figsize=(10, 4))
    b1 = ax.bar(x - width, macro_f1s,  width, label='Macro F1',  color='#1565C0', alpha=0.85)
    b2 = ax.bar(x,         change_f1s, width, label='Change F1', color='#E53935', alpha=0.85)
    b3 = ax.bar(x + width, auc_rocs,   width, label='AUC-ROC',   color='#2E7D32', alpha=0.85)
    for bars in [b1, b2, b3]:
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(short)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title('Ablation: Variant Comparison (tuned threshold)')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'ablation_chart.png', dpi=150)
    plt.show()

    # ── Per-difficulty change-F1 by variant ──
    fig, ax = plt.subplots(figsize=(10, 4))
    difficulties  = ['easy', 'medium', 'hard']
    diff_colors   = ['#43A047', '#FB8C00', '#E53935']
    for i, diff in enumerate(difficulties):
        vals = [ablation[v]['per_difficulty'].get(diff, {}).get('change_f1', float('nan'))
                for v in variants]
        bars = ax.bar(x + (i - 1) * width, vals, width, label=diff,
                      color=diff_colors[i], alpha=0.85)
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(short)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Change-class F1')
    ax.set_title('Per-Difficulty Change F1 — does the ensemble degrade more gracefully?')
    ax.legend(title='Difficulty')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'ablation_per_difficulty.png', dpi=150)
    plt.show()

## 5. Document-Level Error Analysis

In [ ]:
doc_eval_path = RESULTS_DIR / 'document_eval_results.json'
if not doc_eval_path.exists():
    print('Run: python evaluation/document_eval.py')
else:
    with open(doc_eval_path) as f:
        doc_eval = json.load(f)

    models = list(doc_eval.keys())
    metrics = ['doc_precision', 'doc_recall', 'doc_f1']
    labels  = ['Precision', 'Recall', 'F1']

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(models))
    width = 0.25
    colors = ['#1565C0', '#2E7D32', '#E53935']
    for i, (metric, label, color) in enumerate(zip(metrics, labels, colors)):
        vals = [doc_eval[m][metric] for m in models]
        bars = ax.bar(x + (i-1)*width, vals, width, label=label, color=color, alpha=0.85)
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title('Document-Level Evaluation\n(operationally relevant: did we find the right boundary?)')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'doc_eval_chart.png', dpi=150)
    plt.show()

    # ── Per-difficulty document-level F1 ──
    if 'per_difficulty' in doc_eval[models[0]]:
        fig, ax = plt.subplots(figsize=(8, 4))
        difficulties = ['easy', 'medium', 'hard']
        diff_colors  = ['#43A047', '#FB8C00', '#E53935']
        for i, diff in enumerate(difficulties):
            vals = [doc_eval[m]['per_difficulty'].get(diff, {}).get('doc_f1', float('nan'))
                    for m in models]
            bars = ax.bar(x + (i - 1) * width, vals, width, label=diff,
                          color=diff_colors[i], alpha=0.85)
            ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel('Document-level F1')
        ax.set_title('Per-Difficulty Document F1\n(easy = topic shift helps; hard = pure stylometry only)')
        ax.legend(title='Difficulty')
        ax.spines[['top','right']].set_visible(False)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'doc_eval_per_difficulty.png', dpi=150)
        plt.show()